# **UrbanStay: прогнозирование отмен бронирований и оценка экономического эффекта**

- Дайте название проекту, запишите его в блоке Markdown выше.
- Опишите, как вы поняли постановку задачи.
- Опишите данные для решения задачи.

## 1.1 — Название проекта

**UrbanStay: прогнозирование отмен бронирований и оценка экономического эффекта**

## 1.2 — Постановка задачи

UrbanStay — сеть отелей среднего сегмента, которая сталкивается с частыми отменами бронирований незадолго до заселения. По информации заказчика, отменяется более 30% бронирований, а загрузка отелей составляет 60–70% вместо запланированных 80% и выше.

Поздние отмены приводят к простою номеров, потере дохода и затрудняют планирование работы персонала и закупок. Сейчас компания компенсирует отмены с помощью овербукинга, однако такой подход создаёт риск нехватки номеров и дополнительных расходов на размещение гостей.

Цель проекта — разработать модель, которая оценивает вероятность отмены бронирования и помогает заранее выявлять рискованные брони. На основе этих оценок компания сможет предлагать клиентам альтернативные условия или заблаговременно искать новых постояльцев.

Помимо качества прогнозирования, необходимо оценить экономическую эффективность решения: рассчитать прирост дохода, изменение доли отмен и изменение загрузки отеля.

Согласно требованиям итогового задания, необходимо достичь следующих показателей:

- доля отмен после применения модели - 10%;
- снижение загрузки отеля - не более 8%;
- относительный прирост дохода - не менее 50%.

Оценка будет выполнена на тестовой выборке в рамках расчётного сценария задания. Она отражает ожидаемый эффект при принятых допущениях, а не фактически измеренный результат внедрения.

## 1.3 — Описание данных

Для решения задачи предоставлены две таблицы в базе данных PostgreSQL. Загрузка будет выполнена SQL-запросами с использованием SQLAlchemy и pandas.

**Таблица `hotel_bookings` - история бронирований.**

По описанию источника, таблица содержит 35 341 бронирование. Каждая строка соответствует отдельной брони и включает:

- идентификатор и дату оформления бронирования;
- канал продаж;
- количество взрослых и детей;
- сведения о повторных обращениях и предыдущих бронированиях клиента;
- стоимость бронирования и количество дней до заселения;
- количество будних и выходных ночей;
- тип питания и категорию номера;
- наличие парковки и количество специальных пожеланий;
- статус бронирования `booking_status`.

**Таблица `hotel_reviews` - отзывы о проживании.**

По описанию источника, таблица содержит 25 177 отзывов. Для каждого отзыва указаны идентификаторы клиента и бронирования, дата публикации, оценка проживания от 1 до 5 и текст комментария.

Отзывы будут использоваться как источник информации о предыдущем опыте клиента. Для каждого бронирования необходимо подобрать последний доступный отзыв этого же клиента, опубликованный до даты оформления брони. Использование отзывов, появившихся позднее, приведёт к утечке информации.

Фактические размеры таблиц, типы данных, доступные ключи связи, пропуски, дубликаты и аномалии будут проверены после загрузки.

## Перевод бизнес-задачи на язык машинного обучения

- Напишите, что конкретно будете делать с точки зрения Data Science: расскажите, какие модели и методы вы планируете применить для решения задачи.

## 2.1 — Тип задачи и целевой признак

Задача относится к бинарной классификации. Для каждого бронирования модель должна оценить вероятность его отмены.

Целевой признак - `booking_status`:

- `1` - `отказ_брони`: бронирование отменено;
- `0` - `нет_отказа`: бронирование не отменено.

На вход модели будут поступать характеристики бронирования и сведения о предыдущем опыте клиента, доступные на момент оформления брони. На выходе модель будет возвращать вероятность отмены. Для получения итогового класса будет подобран порог классификации.

В предоставленном описании целевой переменной не выделены отдельно поздние отмены. Поэтому модель будет прогнозировать факт отмены бронирования, а не непосредственно отмену за 12–24 часа до заселения.

## 2.2 — Подготовка данных и создание признаков

Подготовка данных будет включать следующие действия:

1. Загрузку таблиц из PostgreSQL и исследовательский анализ числовых и категориальных признаков.
2. Проверку пропусков, дубликатов, типов данных и аномальных значений.
3. Объединение бронирований с последним предшествующим отзывом соответствующего клиента.
4. Создание не менее трёх новых признаков на основе бронирований.

   **ВСТАВИТЬ ПРИЗНАКИ КОТОРЫЕ ПО ИТОГУ ВЫБЕРУ**
5. Преобразование текстов отзывов в числовые признаки с помощью TF-IDF.
6. Подготовку числовых и категориальных признаков с учётом требований выбранных моделей.

Данные будут разделены на обучающую, калибровочную и тестовую выборки в пропорции 60% / 20% / 20%. Конкретный способ разделения будет определён после анализа временной структуры данных.

Обучаемые преобразования, включая заполнение пропусков, кодирование категорий и TF-IDF, будут настраиваться только на обучающей части соответствующего разбиения. Тестовая выборка будет сохранена для финальной оценки выбранного решения.

## 2.3 — Обучение и выбор модели

Для решения задачи планируется сравнить две модели градиентного бустинга:

- `CatBoostClassifier`;
- `LGBMClassifier`.

В качестве простой отправной точки будет использован `DummyClassifier`.

Гиперпараметры моделей будут оптимизированы с помощью Optuna. Для каждой модели будет настроено не менее трёх гиперпараметров. Оценка будет проводиться на трёх фолдах кросс-валидации с размером валидационной части 2000 наблюдений.

Основной критерий выбора модели — Incremental Revenue (IR), то есть прирост расчётного дохода благодаря применению модели. Дополнительно будут рассмотрены матрица ошибок и метрики классификации.

После выбора лучшей модели будет выполнена калибровка вероятностей. Схема калибровки должна исключать обучение калибратора на прогнозах для объектов, использованных при обучении базовой модели, без внутреннего разделения данных.

На калибровочной выборке будет подобран порог классификации, максимизирующий IR. Затем зафиксированное решение будет оценено на тестовой выборке. Для проверки стабильности будут сопоставлены матрицы ошибок и IR на калибровочной и тестовой выборках.

## 2.4 — Оценка экономической эффективности

Экономическая оценка будет основана на сравнении расчётных сценариев до и после применения модели.

Основная метрика:

**IR = доход после применения модели − доход до применения модели.**

В расчётах будут использованы значения, заданные в условии:

| Параметр | Значение |
|---|---:|
| Средний доход от бронирования `AvgRev` | 64 500 ₽ |
| Упущенный доход при отмене `LostRev` | 64 500 ₽ |
| Стоимость ложного предсказания отмены `CostFP` | 7 000 ₽ |
| Доход от повторной продажи номера `PerRebooking` | 45 000 ₽ |

Положительным классом считается отмена бронирования. Поэтому ложное срабатывание FP означает, что модель предсказала отмену, но клиент не отменил бронь, а ошибка FN — что модель пропустила фактическую отмену.

На тестовой выборке будут рассчитаны:

- доход до и после применения модели;
- абсолютный и относительный IR;
- доля отмен до и после применения модели и её относительное изменение;
- загрузка отеля до и после применения модели и её относительное изменение.

Формулы и допущения расчётного сценария будут явно описаны. Полученные результаты будут сопоставлены с целевыми показателями задания.


## 2.5 — Интерпретация и рекомендации

Для интерпретации выбранной модели планируется использовать SHAP. Анализ позволит выделить десять наиболее значимых признаков и оценить, как их значения связаны с прогнозируемым риском отмены.

Влияние признаков на предсказания не будет трактоваться как доказанная причинно-следственная связь. Численные утверждения о размере влияния будут приводиться только при наличии соответствующих расчётов.

Итоговые выводы будут оформлены в двух разделах:

- **«Проделанная работа»** - подготовка данных, созданные признаки, сравнение моделей, лучшие гиперпараметры, порог и результаты оценки;
- **«Бизнес-выводы»** - достижение целевых показателей, экономическая интерпретация и 2–3 конкретные рекомендации для сети отелей.

## Загрузка необходимых библиотек

- Загрузите всё, что планируете использовать для выполнения проекта.

## 3.1 Создание и проверка окржения

In [ ]:
import sys
import subprocess

# фиксируем версии библиотек для работы с данными и визуализации
packages = {
    "numpy": "1.24.4",
    "pandas": "2.2.3",
    "scipy": "1.13.1",
    "matplotlib": "3.9.2",
    "seaborn": "0.13.2",

    # подключение к базе данных PostgreSQL
    "SQLAlchemy": "2.0.36",
    "psycopg2-binary": "2.9.9",

    # предобработка, текстовые признаки, модели и калибровка
    "scikit-learn": "1.5.2",
    "catboost": "1.2.7",
    "lightgbm": "4.5.0",

    # оптимизация гиперпараметров
    "optuna": "4.0.0",

    # интерпретация моделей и совместимые зависимости
    "shap": "0.46.0",
    "numba": "0.60.0",
    "llvmlite": "0.43.0",
}

# устанавливаем пакеты в окружение текущего ядра
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--only-binary=:all:",
    *[f"{name}=={version}" for name, version in packages.items()]
])

# проверяем зависимости всех установленных пакетов
subprocess.check_call([
    sys.executable, "-m", "pip", "check"
])

No broken requirements found.


0

## 3.2 Импорт необходимых библилотек

In [2]:
# работа с данными
import pandas as pd                          # таблицы, даты, агрегации и объединение данных
import numpy as np                           # массивы, вычисления и поиск оптимального порога

# подключение к базе данных
from sqlalchemy import create_engine, text   # подключение к PostgreSQL и SQL-запросы
from sqlalchemy.engine import URL            # создание адреса подключения с учётом спецсимволов пароля

# визуализация
import matplotlib.pyplot as plt              # графики распределений, метрик и результатов
import seaborn as sns                        # статистические графики для EDA

# разделение данных и кросс-валидация
from sklearn.model_selection import (
    train_test_split,                        # разделение данных на обучение, калибровку и тест
    TimeSeriesSplit,                         # временная кросс-валидация с заданным размером валидации
)

# предобработка и объединение признаков
from sklearn.compose import ColumnTransformer  # отдельная обработка числовых, категориальных и текстовых данных
from sklearn.pipeline import Pipeline          # обучение преобразований внутри каждого фолда без утечки
from sklearn.impute import SimpleImputer       # заполнение пропусков
from sklearn.preprocessing import (
    OneHotEncoder,                           # кодирование категорий
    FunctionTransformer,                     # включение собственных преобразований в pipeline
)

# обработка текстов
from sklearn.feature_extraction.text import TfidfVectorizer  # преобразование отзывов в признаки TF-IDF
from scipy import sparse                     # работа с разреженными матрицами текстовых признаков

# модели классификации
from sklearn.dummy import DummyClassifier    # простая отправная точка для сравнения моделей
from catboost import CatBoostClassifier      # прогноз вероятности отмены с помощью CatBoost
from lightgbm import LGBMClassifier           # прогноз вероятности отмены с помощью LightGBM

# калибровка вероятностей
from sklearn.calibration import (
    CalibratedClassifierCV,                  # калибровка вероятностей модели
    CalibrationDisplay,                      # графическое сравнение вероятностей с частотой отмен
)

# метрики классификации и качества вероятностей
from sklearn.metrics import (
    confusion_matrix,                        # значения TN, FP, FN и TP для расчёта IR
    ConfusionMatrixDisplay,                  # визуализация матрицы ошибок
    classification_report,                   # precision, recall и F1 для каждого класса
    roc_auc_score,                           # оценка ранжирования бронирований по риску
    average_precision_score,                 # оценка precision-recall при дисбалансе классов
    brier_score_loss,                        # оценка точности предсказанных вероятностей
    log_loss,                               # оценка вероятностей с усиленным штрафом за уверенные ошибки
)

# оптимизация гиперпараметров
import optuna                                # поиск параметров моделей по максимальному IR

# интерпретация моделей
import shap                                  # оценка вклада признаков в предсказания

# отображение результатов в ноутбуке
from IPython.display import display          # вывод таблиц и результатов

## 3.3 Фиксирование констант, настроек отоброжения таблиц и графиков

In [3]:
# Настройки графиков
sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

# Настройки отображения датафреймов
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.3f}".format)

In [4]:
# Показываем все столбцы и полный текст внутри ячеек
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# В длинных таблицах показываем до 100 строк
pd.set_option('display.max_rows', 100)

# Не переносим столбцы при текстовом выводе таблиц
pd.set_option('display.expand_frame_repr', False)

In [5]:
# Фиксируем случайность для воспроизводимости результатов
RS = 42

## Этап 1: подготовка данных

### Загрузка данных

- Все данные о работе сети отелей хранятся в базе данных на сервере. Для работы с ними в Python используйте библиотеку SQLAlchemy, чтобы выгрузить данные SQL-запросом.

- Сделайте выводы о проделанной работе.

Параметры для подключения к данным PostgreSQL:

- **Хост:** rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net

- **База данных:** data-scientist-hotels

- **Порт:** 6432

- **Пользователь:** praktikum_student

- **Пароль:** Sdf4$2;d-d30pp

Названия нужных вам таблиц — `hotel_bookings` (таблица бронирований) и `hotel_reviews` (таблица отзывов).

### Исследовательский анализ данных и предобработка

- Проведите EDA, используя один из следующих инструментов на ваш выбор:

    - библиотеку pandas;
    - SQL-запросы.
    
- EDA должен включать:
    - аналитическое исследование;
    - графическое исследование для количественных данных и для категориальных данных.

- Сделайте выводы о выбросах, пропусках, дубликатах и других аномалиях  в каждой таблице.
- Проведите необходимую предобработку данных.

### Объединение таблиц

- Соедините таблицу отзывов с таблицей бронирований. Руководствуйтесь правилом: для каждой даты бронирования в столбце `booking_date` нужно взять комментарий и оценку клиента по предыдущей ближайшей дате отзывов, указанной в столбце `review_date`. Это позволит связать обратную связь клиента с конкретным периодом проживания, что критически важно для анализа удовлетворённости гостей.

- Сделайте выводы о том, как прошло объединение и что получилось в результате.

### Создание новых признаков

- Создайте не менее трёх признаков на основе данных о брони.

- Создайте новые признаки на основе текстов отзывов, используя один из методов векторизации. Решение о количестве таких признаков примите самостоятельно.

- Сделать выводы о созданных признаках.

### Анализ итоговой таблицы

- Проанализируйте итоговую таблицу.

- Сделайте выводы о данных для моделирования.

## Этап 2: моделирование

### Обучение и оптимизация модели

- Выберите и обучите не менее двух моделей из списка:

  - Random Forest;

  - CatBoost;

  - LightGBM;

  - XGBoost;

  - любую другую на ваш выбор.

- Разделите данные на три выборки:
  - Обучающую — для обучения моделей (60% данных);
  - Калибровочную — для проведения калибровки модели (20% данных);
  - Тестовую — для финальной оценки качества выбранной модели (20% данных).

- Проведите кросс-валидацию, используйте три фолда. Размер выборки для валидации — 2000.

- Оптимизируйте гиперпараметры с помощью Optuna. Количество гиперпараметров — не менее трёх. Ключевая метрика для оптимизации — Incremental Revenue.

- Сделайте выводы о том, какая модель показала лучшие результаты.

### Калибровка модели и пересчёт результатов

- Выбрав модель, откалибруйте её на полной тренировочной выборке, чтобы повысить обобщающую способность и адаптировать предсказания под бизнес-задачи.

- Сделайте о результатах калибровки модели.

### Поиск порога классификации

- Используя откалиброванную модель и калибровочную выборку, найдите порог классификации, при котором достигается максимальный Incremental Revenue.

- Сделайте выводы о пороге классификации.

### Анализ матрицы классификаций

Оцените стабильность модели на тестовых данных.
- Постройте:
    - матрицу ошибок на калибровочных данных;
    - матрицу ошибок на тестовых данных.

- Посчитайте IR на калибровочных и на тестовых данных.

- Сделайте вывод о стабильности модели.

### Фиксирование итоговой модели

- Зафиксисруйте лучшую модель и найденный порог.


### Анализ важности признаков

- Оцените важность признаков с помощью любого подходящего инструмента:
  - feature_importances;
  - SHAP;
  - встроенной в модель собственной функции оценки важности.

- Сделайте выводы о влиянии признаков на целевую переменную.

## Этап 3: расчёт экономической эффективности модели

Оцените, насколько выгодно внедрять выбранную модель в работу отеля. Для этого нужно выяснить, какой экономический эффект даёт модель и укладываются ли ключевые метрики в заданный уровень.

Если расчёты покажут, что какой-либо показатель не достигает необходимого уровня, то это сигнал к доработке модели. Возможно, вам нужно пересмотреть порог классификации, добавить новые признаки, поменять модель, по-другому предобработать исходные данные — экспериментируйте!

- Шаг 1: подготовка данных. Подготовьте данные для расчётов. Данные для показателей до внедрения модели рассчитывайте с использованием тестовых данных `y_test`, данные после внедрения получите с помощью предсказаний модели `y_pred`.

- Шаг 2: расчёт показателей до и после внедрения модели. Вычислите:
  - Долю отмен бронирования до и после внедрения модели;
  - Загрузку отеля до и после внедрения модели;
  - IR.

- Шаг 3: расчёт динамики показателей. Вычислите:
  - Динамику доли отмен бронирования;
  - Динамику загрузки отеля;
  - Относительный IR — на сколько процентов `IR_после` выше, чем `IR_до`.


Ваша модель должна достигнуть следующих результатов:

- Доля отмен после внедрения модели — 10%

- Загрузка отеля не должна уменьшиться больше чем на 8% после внедрения модели.

- Относительный IR должен составить не менее 50%.

Сделайте выводы о том, получилось ли достичь целевых показателей для бизнеса.


## Этап 4: выводы по проекту

Выводы должны состоять из двух логически связанных разделов:

- «Проделанная работа» — описание этапов и решений;

- «Бизнес‑выводы» — интерпретация результатов и рекомендации.

В каждом разделе опишите результаты без избыточной детализации, с опорой на факты и цифры.

### Выводы о проделанной работе

В этом разделе опишите основные этапы проделанной работы по построению модели. Опишите, как проходили следующие шаги:
- Подготовка данных;
- Моделирование;
- Оценка метрик;
- Анализа важности факторов.

### Выводы по анализу эффективности модели

В этом разделе ответьте на вопрос: «Что это значит для бизнеса?» Для этого интерпретируйте результаты вашей работы, дайте им экономическую оценку, а заказчику — рекомендации.

Включите следующие пункты:

- Итоговая оценка достижения цели:
  - Вспомните цель проекта и определите, достигнута ли она. Аргументируйте свой ответ.

- Результаты по ключевым метрикам. Для каждого показателя приведите:
  - Значение до внедрения модели.
  - Значение после внедрения.
  - Изменение в процентах с расчётом по формуле.
  
- Сообщите заказчику, достигнуты ли целевые показатели по метрикам.

- Анализ важности признаков:
  - Опишите для заказчиков основные 10 признаков, влияющих на резкие отмены заказов.
  - Кратко объясните, как они влияют на целевую переменную. Пример такого объяснения: «лояльность клиента снижает риск отмены на 15%».

- Рекомендации для бизнеса:
  - Предложите 2–3 конкретных шага по оптимизации работы сети отелей.